In [1]:
# -------------------------------------------------------
# SECTION 1: Import Required Libraries
# -------------------------------------------------------

import pandas as pd
from neo4j import GraphDatabase

In [2]:
# -------------------------------------------------------
# SECTION 2: Load the dataset files
# -------------------------------------------------------

customers = pd.read_csv("../data/dim_customer.csv")
products = pd.read_csv("../data/dim_product.csv")
orders = pd.read_csv("../data/fact_order.csv")

# Preview data
print(customers.head())
print(products.head())
print(orders.head())

   customer_id             customer_name   region  age  gender Customer_since
0            1                Robin Ward    North   51    Male     2022-07-26
1            2            Joseph Chapman     East   21  Female     2023-04-21
2            3  Dr. Alicia Henderson DVM    North   19    Male     2024-03-22
3            4                Laura King  Central   20    Male     2024-08-03
4            5             Brian Wheeler    South   18  Female     2023-02-07
   product_id product_name     category  unit_price  \
0        1001         Huge      Apparel       99.32   
1        1002      Address    Furniture      290.67   
2        1003       Almost  Electronics       37.75   
3        1004        Month       Beauty       75.32   
4        1005         Plan       Beauty      332.12   

                      supplier  
0                    Owens Inc  
1              Romero and Sons  
2  Williams, Shah and Mitchell  
3            Williams-Stephens  
4                   Gibson Inc  
   

In [3]:
# -------------------------------------------------------
# SECTION 3: Connect Python to Neo4j AuraDB
# -------------------------------------------------------

uri = "neo4j+s://c2f51abb.databases.neo4j.io"
username = "c2f51abb"
password = "0KTQ6e0BYA1mQEoB2NcIqgb6GXTmzRE_rrgKAH3GsIo"

driver = GraphDatabase.driver(uri, auth=(username, password))

print("Connected to Neo4j Aura")

Connected to Neo4j Aura


In [4]:
with driver.session() as session:
    result = session.run("RETURN 'Connection successful!' AS message")
    
    for record in result:
        print(record["message"])

Connection successful!


In [5]:
# -------------------------------------------------------
# Create Customer nodes
# -------------------------------------------------------

def create_customer(tx, row):

    query = """
    MERGE (c:Customer {customer_id:$customer_id})

    SET c.name = $name,
        c.region = $region,
        c.age = $age,
        c.gender = $gender,
        c.customer_since = $customer_since
    """

    tx.run(query,
           customer_id=row["customer_id"],
           name=row["customer_name"],
           region=row["region"],
           age=row["age"],
           gender=row["gender"],
           customer_since=row["Customer_since"])

In [6]:
# -------------------------------------------------------
# Create Product nodes
# -------------------------------------------------------

def create_product(tx, row):

    query = """
    MERGE (p:Product {product_id:$product_id})

    SET p.name = $name,
        p.unit_price = $price
    """

    tx.run(query,
           product_id=row["product_id"],
           name=row["product_name"],
           price=row["unit_price"])

In [7]:
# -------------------------------------------------------
# Create Category and Supplier relationships
# -------------------------------------------------------

def create_product_relationships(tx, row):

    query = """
    MERGE (p:Product {product_id:$product_id})

    MERGE (c:Category {name:$category})
    MERGE (s:Supplier {name:$supplier})

    MERGE (p)-[:BELONGS_TO]->(c)
    MERGE (p)-[:SUPPLIED_BY]->(s)
    """

    tx.run(query,
           product_id=row["product_id"],
           category=row["category"],
           supplier=row["supplier"])

In [8]:
# -------------------------------------------------------
# Create Orders and connect them to customers/products
# -------------------------------------------------------

def create_order(tx, row):

    query = """
    MERGE (o:Order {order_id:$order_id})

    SET o.order_date = $order_date,
        o.quantity = $quantity,
        o.sales_amount = $sales

    WITH o

    MATCH (c:Customer {customer_id:$customer_id})
    MATCH (p:Product {product_id:$product_id})

    MERGE (c)-[:PLACED_ORDER]->(o)
    MERGE (o)-[:CONTAINS_PRODUCT]->(p)
    """

    tx.run(query,
           order_id=row["order_id"],
           order_date=row["order_date"],
           quantity=row["quantity"],
           sales=row["sales_amount"],
           customer_id=row["customer_id"],
           product_id=row["product_id"])

In [9]:
print(len(customers))
print(len(products))
print(len(orders))

1000
100
2000


In [10]:
# -------------------------------------------------------
# Insert data into Neo4j graph
# -------------------------------------------------------

with driver.session() as session:

    session.run("""
    UNWIND $rows AS row
    MERGE (c:Customer {customer_id: row.customer_id})
    SET c.name = row.customer_name,
        c.region = row.region,
        c.age = row.age,
        c.gender = row.gender,
        c.customer_since = row.Customer_since
    """, rows=customers.to_dict("records"))

    session.run("""
    UNWIND $rows AS row
    MERGE (p:Product {product_id: row.product_id})
    SET p.name = row.product_name,
        p.unit_price = row.unit_price
    """, rows=products.to_dict("records"))

    session.run("""
    UNWIND $rows AS row
    MERGE (p:Product {product_id: row.product_id})
    MERGE (c:Category {name: row.category})
    MERGE (s:Supplier {name: row.supplier})
    MERGE (p)-[:BELONGS_TO]->(c)
    MERGE (p)-[:SUPPLIED_BY]->(s)
    """, rows=products.to_dict("records"))

    session.run("""
    UNWIND $rows AS row
    MERGE (o:Order {order_id: row.order_id})
    SET o.order_date = row.order_date,
        o.quantity = row.quantity,
        o.sales_amount = row.sales_amount

    WITH o, row
    MATCH (c:Customer {customer_id: row.customer_id})
    MATCH (p:Product {product_id: row.product_id})

    MERGE (c)-[:PLACED_ORDER]->(o)
    MERGE (o)-[:CONTAINS_PRODUCT]->(p)
    """, rows=orders.to_dict("records"))

print("Knowledge graph created successfully")

Knowledge graph created successfully
